In [1]:
from diffforms import *
from sympy import sin, cos, sqrt #Sympy functions for things

# Abreviate long commands
PD = PartialDerivative
CD = CovariantDerivative
CT = Contract
PI = PermuteIndices

In [20]:
M = Manifold("M", 4, signature=-1) 

coords = t,r,theta,phi = symbols(r"t r \theta \phi",real=True) # Coordinates are sympy symbols
mass = constants("M") # constants works the same as symbols but gives constants

M.set_coordinates(coords)

basis = dt,dr,dtheta,dphi = M.get_basis()
vects = vt,vr,vtheta,vphi = M.get_vectors()

F = Function(r"f",real=True,positive=True)(r)
G = Function(r"g",real=True,positive=True)(r)
Rho = Function(r"\rho",real=True,positive=True)(r)


schwarz_sol = {N: sqrt(1-mass/r), F: 1/sqrt(1-mass/r)}

e_I = e0,e1,e2,e3 = F*dt, G*dr, Rho*dtheta, Rho*sin(theta)*dphi

M.set_frame(e_I)

g_DD = M.get_metric() # Computes metric from frame
g_UU = M.get_inverse_metric() # and its inverse

G_UDD = M.get_christoffel_symbols()

print("Metric:")
display(g_DD)
print("Christoffel Symbols:")
G_UDD

Metric:


$(- f^{2})d\left(t\right) \otimes d\left(t\right)+(g^{2})d\left(r\right) \otimes d\left(r\right)+(\rho^{2})d\left(\theta\right) \otimes d\left(\theta\right)+(\rho^{2} \sin^{2}{\left(\theta \right)})d\left(\phi\right) \otimes d\left(\phi\right)$

Christoffel Symbols:


$(\frac{\frac{d}{d r} f}{f})\partial_{t} \otimes d\left(r\right) \otimes d\left(t\right)+(\frac{\frac{d}{d r} g}{g})\partial_{r} \otimes d\left(r\right) \otimes d\left(r\right)+(\frac{\frac{d}{d r} \rho}{\rho})\partial_{\theta} \otimes d\left(r\right) \otimes d\left(\theta\right)+(\frac{\frac{d}{d r} \rho}{\rho})\partial_{\phi} \otimes d\left(r\right) \otimes d\left(\phi\right)+(\frac{1}{\tan{\left(\theta \right)}})\partial_{\phi} \otimes d\left(\theta\right) \otimes d\left(\phi\right)+(\frac{\frac{d}{d r} f}{f})\partial_{t} \otimes d\left(t\right) \otimes d\left(r\right)+(\frac{\frac{d}{d r} \rho}{\rho})\partial_{\theta} \otimes d\left(\theta\right) \otimes d\left(r\right)+(\frac{\frac{d}{d r} \rho}{\rho})\partial_{\phi} \otimes d\left(\phi\right) \otimes d\left(r\right)+(\frac{1}{\tan{\left(\theta \right)}})\partial_{\phi} \otimes d\left(\phi\right) \otimes d\left(\theta\right)+(\frac{f \frac{d}{d r} f}{g^{2}})\partial_{r} \otimes d\left(t\right) \otimes d\left(t\right)+(- \frac{\rho

In [21]:
R_UDDD = M.get_riemann_curvature_tensor()

# Ricci tensor from contraction of Riemman
R_DD = CT(R_UDDD,(0,2))

# Ricci scalar from contraction of Ricci
R = CT(R_DD*g_UU,(0,2),(1,3)).simplify()

Lambda = constants("Λ")
# Einstein Tensor
G_DD = R_DD - Number(1,2)*g_DD*R - g_DD*Lambda
G_DD = G_DD.simplify()
display(G_DD.subs(Lambda,0))

$(\frac{f^{2} \left(- 2 \rho g \frac{d^{2}}{d r^{2}} \rho + 2 \rho \frac{d}{d r} \rho \frac{d}{d r} g + g^{3} - g \left(\frac{d}{d r} \rho\right)^{2}\right)}{\rho^{2} g^{3}})d\left(t\right) \otimes d\left(t\right)+(\frac{\rho \left(\rho g \frac{d^{2}}{d r^{2}} f - \rho \frac{d}{d r} f \frac{d}{d r} g + f g \frac{d^{2}}{d r^{2}} \rho - f \frac{d}{d r} \rho \frac{d}{d r} g + g \frac{d}{d r} \rho \frac{d}{d r} f\right)}{f g^{3}})d\left(\theta\right) \otimes d\left(\theta\right)+(\frac{\rho \left(\rho g \frac{d^{2}}{d r^{2}} f - \rho \frac{d}{d r} f \frac{d}{d r} g + f g \frac{d^{2}}{d r^{2}} \rho - f \frac{d}{d r} \rho \frac{d}{d r} g + g \frac{d}{d r} \rho \frac{d}{d r} f\right) \sin^{2}{\left(\theta \right)}}{f g^{3}})d\left(\phi\right) \otimes d\left(\phi\right)+(\frac{2 \rho \frac{d}{d r} \rho \frac{d}{d r} f + f \left(- g^{2} + \left(\frac{d}{d r} \rho\right)^{2}\right)}{\rho^{2} f})d\left(r\right) \otimes d\left(r\right)$

In [35]:
tmp1 = (CT(G_DD*vt*vt,(0,2),(1,3)).simplify().subs(Lambda,0)*G**2/F**2).expand()
tmp2 = (CT(G_DD*vr*vr,(0,2),(1,3)).simplify().subs(Lambda,0)).expand()

display_no_arg( tmp1 + tmp2 )

<IPython.core.display.Math object>

In [4]:
from sympy import dsolve
Feq = dsolve(CT(G_DD*vt*vt,(0,2),(1,3)).simplify(),F)[0]
Fsol = {Feq.lhs: Feq.rhs}

Neq = dsolve(CT(G_DD.subs(Fsol).simplify()*vr*vr,(0,2),(1,3)).simplify(),N)
FNsol = {Feq.lhs: Feq.rhs, Neq.lhs: Neq.rhs}

display( G_DD.subs(FNsol).simplify() )

display(Feq)
display(Neq)

$0$

Eq(F(r), -sqrt(3)*sqrt(r/(Λ*r**3 + C1 + 3*r)))

Eq(N(r), C2*sqrt(Λ*r**3 + C1 + 3*r)/sqrt(r))